# Notebook 1 · ODEs and physics-informed surrogates

A neural network can fit observations of a dynamical system. The more useful question is whether it can also respect the equation that generated the data.

The example is intentionally small so that every term can be inspected.


## Model problem

The analytic solution of $u'(t) = -u(t)$ with $u(0)=1$ is $u(t)=e^{-t}$.

We train two multilayer perceptrons (MLPs) $\hat u_\theta$ on noisy samples on $[0,2]$:

1. a data-only network that minimises squared error on the measured points;
2. a physics-informed network that also penalises the residual $\hat u'_\theta + \hat u_\theta$ on collocation points.

Initial condition $u(0)=1$ is enforced with a separate penalty.

Qubits default to $\lvert 0\rangle$ in later notebooks; here the state is purely classical real-valued.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Model and residual

The residual uses autograd to differentiate the neural surrogate with respect to its input.

In [2]:
class MLP(nn.Module):
    """Small surrogate for scalar u(t)."""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.net(t)


def ode_residual(model: MLP, t: torch.Tensor) -> torch.Tensor:
    """Compute du/dt + u with autograd."""
    t_req = t.clone().detach().requires_grad_(True)
    u = model(t_req)
    du = torch.autograd.grad(u.sum(), t_req, create_graph=True)[0]
    return du + u

## Training loop

The same function trains the data-only and physics-informed variants by changing the residual weight.

In [3]:
def train_pinn(
    epochs: int = 2500,
    physics_weight: float = 0.35,
    ic_weight: float = 12.0,
) -> MLP:
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=8e-3)
    t_col = torch.linspace(0.0, 2.0, 36).reshape(-1, 1)
    noise = 0.03 * torch.randn_like(t_col)
    targets = torch.exp(-t_col) + noise
    t_phys = torch.linspace(0.0, 2.0, 90).reshape(-1, 1)
    t0 = torch.zeros(1, 1)
    history = {"data": [], "physics": [], "ic": []}
    for step in range(epochs):
        opt.zero_grad()
        pred_col = model(t_col)
        data_loss = torch.mean((pred_col - targets) ** 2)
        res = ode_residual(model, t_phys)
        phys_loss = torch.mean(res**2)
        ic_loss = (model(t0) - 1.0) ** 2
        loss = data_loss + physics_weight * phys_loss + ic_weight * ic_loss
        loss.backward()
        opt.step()
        if step % 100 == 0:
            history["data"].append(float(data_loss.detach()))
            history["physics"].append(float(phys_loss.detach()))
            history["ic"].append(float(ic_loss.detach()))
    return model, history

## Fit both variants

In [4]:
data_only, data_hist = train_pinn(physics_weight=0.0, ic_weight=12.0)
pinn, pinn_hist = train_pinn(physics_weight=0.35, ic_weight=12.0)
t_plot = torch.linspace(0, 2, 240).reshape(-1, 1)
with torch.no_grad():
    pred_data = data_only(t_plot).numpy().ravel()
    pred_pinn = pinn(t_plot).numpy().ravel()
truth = np.exp(-t_plot.numpy().ravel())

## What to notice

The data-only network can fit measured points while still violating the ODE between them. The PINN-style loss anchors the curve toward the equation in regions where the equation is trusted.


In [ ]:
plt.figure(figsize=(6.4, 4.0))
plt.plot(t_plot.numpy(), truth, label="exp(-t)")
plt.plot(t_plot.numpy(), pred_data, "--", label="data-only")
plt.plot(t_plot.numpy(), pred_pinn, "-.", label="PINN-style")
plt.xlabel("t")
plt.ylabel("u")
plt.legend()
plt.tight_layout()
plt.savefig("pinn_exp_decay.png", dpi=110)
plt.show()

## Diagnostics

The residual mean-square below is not a training metric for the data-only model, but it is a useful audit. If two models have similar data error, the lower residual error is usually the more physically consistent surrogate for this toy problem.


In [ ]:
with torch.no_grad():
    data_mse = torch.mean((data_only(t_plot) - torch.exp(-t_plot)) ** 2).item()
    pinn_mse = torch.mean((pinn(t_plot) - torch.exp(-t_plot)) ** 2).item()
data_res = torch.mean(ode_residual(data_only, t_plot) ** 2).item()
pinn_res = torch.mean(ode_residual(pinn, t_plot) ** 2).item()

print(f"data-only MSE to exact solution: {data_mse:.4e}")
print(f"PINN-style MSE to exact solution: {pinn_mse:.4e}")
print(f"data-only residual MSE: {data_res:.4e}")
print(f"PINN-style residual MSE: {pinn_res:.4e}")